# Design: pooled annual differential timeliness

## Frozen scope and source evidence

This attempt covers `annual_pooled_dt` / `annual_pooled_eq1` only. The paper's Equation (1) is on physical PDF page 4 (printed p. 68): continuing-operations EPS divided by the price at the end of the prior period is regressed on a negative-return indicator, Return, and their interaction. Footnote 3 says the default data use raw returns and continuing-operations EPS. The paper lists two alternative annual Return windows but does not identify which one generated the pooled annual estimate. Physical PDF page 5 (printed p. 69) reports annual data available over 1951–2001, an initial sample of 136,887 firm-years, annual trimming at both tails, a final 131,920 firm-years, and a pooled interaction estimate of 0.302. Those counts and the estimate are comparison points, not construction targets.

The baseline will use the first of the two reported alternatives: fiscal-year stock returns. This is a best-effort timing choice because the paper does not identify the pooled fit's window. I define each annual window using actual adjacent Compustat DATADATEs, from the prior date (exclusive) through the current date (inclusive), and compound CRSP daily total returns over that exact interval. A fiscal-year label does not guarantee exactly twelve months when the reporting date changes; the actual interval length will be retained and summarized, not silently normalized to a calendar year.

## Source files and field mapping

The read-only source tables are the exact catalog-listed tables below. The catalog is `output/feasibility_runs/resources/catalog.json`.

| Source | Fields used | Purpose |
| --- | --- | --- |
| `COMP.FUNDA` at `/storage/sas/base/comp/d_na/funda.sas7bdat` | `GVKEY`, `DATADATE`, `FYEAR`, `INDFMT`, `DATAFMT`, `CONSOL`, `POPSRC`, `IID`, `HIID`, `CURCD`, `EPSPX`, `AJEX`, `PRCC_F`, `SICH`, `NAICSH` | Annual EPS, fiscal-period price, actual annual dates, current-header and historical primary issue identifiers, reporting currency, and historical industry codes used only to resolve conflicting format duplicates |
| `COMP.SEC_DPRC` at `/storage/sas/base/comp/d_na/security/sec_dprc.sas7bdat` | `GVKEY`, `IID`, `DATADATE`, `CURCDD` | Security-level daily price currency for the primary Compustat issue at the most recent available date on or before the prior annual DATADATE |
| `COMP.SEC_MTHPRC` at `/storage/sas/base/comp/d_na/security/sec_mthprc.sas7bdat` | `GVKEY`, `IID`, `DATADATE`, `CURCDM` | Monthly issue-level currency fallback when daily currency is unavailable |
| `CRSP.CCMXPF_LINKTABLE` at `/storage/sas/base/crsp/a_ccm/ccmxpf_linktable.sas7bdat` | `GVKEY`, `LIID`, `LPERMNO`, `LINKDT`, `LINKENDDT`, `LINKPRIM`, `LINKTYPE`, `USEDFLAG` | Historical company/security link to CRSP |
| `CRSP.DSI` at `/storage/sas/base/crsp/a_stock/dsi.sas7bdat` | `DATE` | Market trading-day calendar for expected return-window session coverage |
| `CRSP.DSF` at `/storage/sas/base/crsp/a_stock/dsf.sas7bdat` | `PERMNO`, `DATE`, `PRC`, `RET` | Raw daily USD stock price and daily stock total returns |

The paper uses Standard & Poor's Compustat 2001 Primary-Supplementary-Tertiary, Full Coverage, and Research files. The current local annual file is a later merged product, not that historical vintage. I use domestic (`POPSRC='D'`), standard (`DATAFMT='STD'`), consolidated (`CONSOL='C'`) annual company records and include both available industrial and financial-services formats (`INDFMT='INDL'` and `INDFMT='FS'`), with no industry exclusion. Source rows identical on all model inputs and dates are collapsed even if their format labels differ because they yield one identical firm-year. For conflicting format rows, a valid historical `SICH` code from 100 through 9999 classifies 6000–6999 as financial services (`FS`) and other valid SIC codes as industrial (`INDL`). Zero and out-of-range SIC values are treated as unavailable for this classification; this is a code-validity inference, because the provider guide identifies the field as historical SIC but does not give a separate numeric missing-code convention. When `SICH` is unavailable or invalid, the code derives the two-digit sector from the stored historical `NAICSH` code, which the guide describes hierarchically from 2-digit sectors through 6-digit industries; sector 52 indicates `FS`, while another observed sector indicates `INDL`. A firm-year with conflicting duplicate dates or no unique row under that rule is ambiguous and is counted and excluded. The 2004 provider guide defines `SICH` as historical SIC (physical p. 97 / printed p. 47) and `NAICSH` as historical NAICS (physical p. 90 / printed p. 40); it notes that `SICH` is unavailable for some banks and financial institutions. These codes resolve format ambiguity only and do not screen the population. Annual availability, zero codes, and derived NAICS sector availability are tabulated. The mapping is a current-data best effort, not a claim that these filters recreate the 2001 CD exactly.

The provider's Xpressfeed usage guide gives the split-adjusted basic EPS formula as `EPSPX/AJEX` (2011 guide, physical p. 29 / printed p. 21) and says price/share items are delivered unadjusted (physical pp. 107–108 / printed pp. 99–100). The 2004 Research Insight guide defines `EPSPX` as basic EPS before extraordinary items and discontinued operations and describes the 1997 transition from primary to basic EPS (physical pp. 659–660 / printed pp. 609–610). Its `PRCC_F` item is the fiscal-year closing price, including the last available trading close for a suspended issue (physical pp. 674–675 / printed pp. 624–625). This is the closest documented Compustat item to the paper's continuing-operations EPS; the historical primary/basic convention remains a product-vintage limitation.

For the dependent variable, use `EPSPX/AJEX` and an explicitly aligned raw price at the preceding fiscal-year date. Rebase the Compustat adjusted EPS back to the preceding fiscal-date share basis using the same-provider relative factor:

`EPS_on_prior_share_basis = (EPSPX_t / AJEX_t) * AJEX_(t-1)`

This is equivalent to `EPSPX_t * (AJEX_(t-1)/AJEX_t)`. The Compustat Xpressfeed guide illustrates that a split after the prior fiscal date raises that date's AJEX, so this relative ratio converts current EPS to the share basis of the earlier price. It does not use a cumulative adjustment factor from another provider.

For USD reporting EPS, divide that numerator by the absolute CRSP `PRC` quote in USD on the same linked PERMNO at the preceding actual Compustat DATADATE. If the actual date is not a trading date, use the last quote on or before it within seven calendar days. The quote date and its gap from DATADATE are retained and summarized. No CRSP cumulative factor is used; the Compustat relative AJEX ratio places current EPS on the prior fiscal-date share basis of the raw CRSP quote.

For non-USD reporting EPS, use the prior Compustat annual raw `PRCC_F` only when the historical primary issue's security-price currency (`CURCDD`/`CURCDM`) at the prior price date explicitly matches the current EPS `CURCD`. The local `COMP.FUNDA.IID` label is “Header Primary Global Issue ID”; `COMP.FUNDA.HIID` is labeled “Historic Primary Global Issue ID.” Use `HIID` for date-specific issue matching and retain `IID` as a source diagnostic. Then:

`eps_price = EPS_on_prior_share_basis / PRCC_F_(t-1)`

This uses the previous fiscal-date price, not the current year's price. The same EPS rebasing and fiscal-date denominator definition apply to every currency. The price source differs by currency: CRSP's documented USD quote for USD EPS, or Compustat's annual unadjusted `PRCC_F` where contemporaneous historical-primary-HIID security-price currency evidence matches non-USD EPS. No equality between absolute CRSP and Compustat adjustment-factor bases is assumed. The corrected currency audit found no matched `SEC_DPRC` price-currency records before FY 1983; `SEC_MTHPRC` records first match in FY 1961. All requested endpoints in FY 1951–1956 have USD reporting currency. FY 1957–1960 include some non-USD requests without issue-currency metadata; those ratios remain unavailable under the selected rules. These request counts include current and prior endpoints and are diagnostics, not analysis-sample counts.

The bounded audit `coverage.sas` matched annual historical-primary-HIID endpoints to Compustat security daily and monthly currency records; its corrected summary is in `runtime/coverage/coverage_hiid.lst`. For each annual `HIID`, select the most recent daily `CURCDD` at or before the prior actual date within seven days; if unavailable, use the most recent monthly `CURCDM` at or before that date within one monthly interval. If both are available and conflict, price currency is unavailable. Keep a non-USD ratio only when current EPS `CURCD` equals the selected security-price currency and the historical primary `HIID` is unchanged. Retain selected source dates and gaps. Missing or mismatched currencies remain unavailable; no currency conversion, country exclusion, or zero-fill is applied. The security-price descriptor is issue-level, while EPS and annual `PRCC_F` are company-level values associated with the historical primary issue. A bounded synthetic native SAS-to-Stata handoff confirmed the prior-share-basis calculation using `EPSPX_t=2.4`, `AJEX_t=1.5`, `AJEX_(t-1)=3`, prior price 24, and unequal current comparison price 17: the retained ratio is 0.20 (`4.8/24`), not the current-price ratio `4.8/17`.

For the CRSP link, require `USEDFLAG=1`, `LINKPRIM` in `P`/`C`, and `LINKTYPE` in `LC`/`LU`; use the historical primary `HIID` to match `LIID` whenever the dated issue ID is identified. The 2013 CRSP/Compustat guide (physical pp. 8–9, 15, and 17) documents these link fields, `P`/`C` as primary links, `LC` as researched links, `LU` as unresearched CUSIP links, and `USEDFLAG=1` as a used link. The guide also documents dummy `LIID` values ending in `X` for date ranges where company data exists but Compustat has no identified security (physical pp. 19 and 21 / printed pp. 15 and 17). Accept such a dummy only over its actual link dates, with the same historical `HIID` across adjacent annual records and an unambiguous PERMNO. A continuous coalesced link segment must cover both annual report-date endpoints. Final route flags count original valid CCM rows that overlap the actual prior/current date interval; separate history flags show whether either route appears anywhere in that period's coalesced link segment. This keeps link construction and route attribution distinct. These provider-dated X rows explain early company history; no current CUSIP or name-similarity fallback is used.

Before construction, a bounded native `PROC CONTENTS`/`PROC FREQ` and date-window audit examined the exact catalog table `CRSP.CCMXPF_LINKTABLE` (92,711 rows). The local table has no missing or special-missing `LINKDT` rows; 14,750 rows use the open-end `.E` code, which the date join treats as an open end. The initial check matching `LINKTABLE.LIID` only to `FUNDA.IID` incorrectly found no 1951–1961 endpoint links because `IID` is a header field. Rechecking historical primary `HIID` and provider-documented X placeholders found date-valid links: the exact table includes used LC, `LINKPRIM=C`, `LIID=00X` intervals beginning in 1950 that map to a single PERMNO. One exact local example is GVKEY 001045: `FUNDA.IID=04`, `FUNDA.HIID=01`, and CCM maps `00X` to PERMNO 21020 through 1962-01-30, followed by `LIID=01` to the same PERMNO from 1962-01-31. The full construction reports the annual recovered sample and distinguishes route rows overlapping each actual return interval from routes found elsewhere in that period's coalesced link history.

For link coverage, match intervals only when `LIID=HIID` or the source row is a documented X placeholder, and only after requiring stable `HIID` at both annual endpoints. Coalesce date intervals per `GVKEY`/`FYEAR`/`HIID`/`PERMNO` only when they overlap or are adjacent and each row matches the direct historical ID or X-placeholder rule. Thus a placeholder-to-identified-IID change can represent a continuous primary security if the PERMNO is the same and there is no date gap. Count candidate PERMNOs after coalescing; ambiguous firm-years with more than one remain excluded. The initial header-IID diagnosis that FY 1951–1961 lacked historical link support is superseded by this date-specific HIID/X correction; no later link is backdated.

I also audited the link predicates separately. Removing `USEDFLAG=1` does not change the annual `LC`/`LU` company-year match counts in the local table; the provider guide describes this flag as identifying a link used in CRSP composite construction, not a general quality score. Widening `LINKTYPE` to include `LN` and `LS` adds 11 same-header-IID firm-years in FY 1951–1961 but does not identify the historical primary issue in the early periods. The guide defines `LN` as a primary link where Compustat has no prices and `LS` as valid only for that security. For the baseline, I retain the standard researched/unresearched `LC`/`LU` criteria.

No current CUSIP or name-similarity fallback is used. The Compustat Research Insight guide describes company CUSIPs with Compustat-assigned synthetic issuer and issue codes (physical pp. 71–72 / printed pp. 21–22). The CRSP Stock and Index Data Descriptions Guide says historical end-of-period CUSIPs were first assigned in 1968 and are unavailable in CRSP name history before then (physical p. 27 / printed p. 23); it distinguishes that item from the latest/header `NCUSIP` at physical p. 28 / printed p. 24. A dated `CRSP.DSENAMES` CUSIP join therefore cannot provide an issue link for the 1951–1961 gap.

CRSP `RET` is the daily holding-period total return (CRSP Stock and Index Data Descriptions Guide, physical pp. 44 and 128). Use the `CRSP.DSI` date calendar to count expected market sessions in `(prior DATADATE, current DATADATE]`. Require DSF records for every expected session, matching first and last observed dates, and valid returns for every session before compounding the annual Return. This detects a missing interior day and a short listing inside an otherwise full-year window; it does not impose an arbitrary 250-session threshold, so a genuinely short actual reporting interval is retained with its true duration and expected-session count. Missing `RET` values and codes below -1 are unavailable, never zero. Intervals that lack full DSF coverage, including an issue delisted before the current reporting date, remain unavailable; the paper does not specify a delisting-return extension, so I do not add `DLRET` beyond its documented daily-`RET` baseline. The bounded synthetic handoff rejects both a ten-session interior gap (9 observed of 10 expected) and a ten-day partial listing (10 observed of 252 expected), while accepting a complete ten-session short actual window.

## Population, lag, and trimming

Build supporting annual records for fiscal-year labels 1950–2001 so fiscal year 1950 can support the first in-scope 1951 lag. Retain actual `DATADATE` values; the end-year label is not used as a substitute for the last reporting date. The analysis period is `FYEAR=1951` through `FYEAR=2001`; the prior price is joined by `GVKEY` and exactly `FYEAR-1`, and its DATADATE must precede the current DATADATE. A gap in annual labels, missing prior record, changed historical primary `HIID`, or unavailable prior price remains missing. The fiscal-year Return uses the two actual DATADATEs as endpoints, preserving the reporting-period interval rather than forcing a twelve-month calendar interval when a fiscal year-end changes. Actual period lengths in days will be retained and tabulated so stub/extended periods are visible.

A bounded follow-up audit of the retained windows shorter than 300 days or longer than 400 days checks all raw annual `GVKEY`/`DATADATE` keys in the exact filtered `COMP.FUNDA` source, independent of their `FYEAR` labels. The audit reuses the final window keys only because A1 and A2 were verified one-for-one on all 139,737 `GVKEY`/`FYEAR` rows and all 50 shared non-route fields, including `DATADATE`, `PREVDATE`, and `RET_DAYS`; the A1 handoff is not used for estimation. The audit does not impose a duration rule. `FYR` (fiscal year-end month), `PDDUR` (period duration), and `APDEDATE` (actual period-end date) are examined as available descriptors. In the filtered historical source, `APDEDATE` is missing and the available provider guide's month-unit definition for `PDDUR` is specific to international companies, so neither field is used to replace the selected `DATADATE` endpoints for this domestic-source baseline.

The audit found no intermediate raw annual `DATADATE` for any of the 974 retained short/long windows (390 under 300 days, 584 over 400 days); `FYR` changes between the endpoints in all 974. This supports adjacent observed annual source dates and rules out an intermediate date lost solely by FYEAR deduplication within the filtered source. Detailed counts, PDDUR values, APDEDATE availability, and the limits of this interpretation are recorded in `review.md`.

Construct `DUM=1` for negative Return and `0` for zero or positive Return; construct `DUM_RETURN=DUM*Return`. On the complete-case annual sample, remove observations outside the 1st/99th percentiles of `eps_price` and Return separately within each fiscal-year label, as described on physical PDF page 5, footnote 4. Ties at a cutoff are handled by the percentile comparisons and may yield an annual retained count that differs from an exact 98 percent. No other winsorization or outlier screen is applied. The sequence and counts for annual source rows, lag matches, currency checks, security links, complete return windows, and each trimming step will be logged, along with distributions of the final regression inputs.

## Construction results and actual coverage

The first full construction used header `FUNDA.IID` as the historical link key and produced 128,774 rows; it was superseded after field definitions and dated-link audits showed that `IID` is a header field. The corrected construction retains 139,737 firm-years in every target year from 1951 through 2001. Annual counts range from 297 in 1951 to 6,118 in 1997. There are 5,910 retained firm-years where header `IID` differs from historical primary `HIID`.

The A1 route flags aggregated direct-ID and X-placeholder evidence across the whole coalesced CCM segment associated with a firm-year. They described coalesced history, not the link rows overlapping that observation's dates. A2 retains these whole-segment counts as `dummy_x_history` and `direct_id_history`, then sets `dummy_x_used` and `direct_id_used` from original valid CCM rows overlapping the inclusive link-support interval from prior to current `DATADATE`. On the final sample, 5,371 observations have an X interval overlapping that support (4,590 X-only and 781 with both routes); 135,147 have direct-HIID overlap (134,366 direct-only and 781 with both). The separate history flags show X anywhere in the coalesced segment for 31,631 observations (130 X-only, 31,501 also with direct-ID history) and direct-ID history for 139,607 (108,106 direct-only plus the same 31,501). Route flags do not determine eligibility, so this correction changes attribution only. The input records and estimates are unchanged.

The SAS attrition table reports 318,384 selected source firm-years in the target years; 292,140 have an adjacent-year lag with ordered actual dates, 285,150 have stable historical `HIID`, 245,820 have EPS and split-factor inputs, 175,859 have one date-valid PERMNO, 164,182 have complete daily-return windows, 186,029 have a matched currency and price path, and 145,055 are complete regression cases before trimming. Among unique pretrim links, 8,756 overlap an X row and 168,323 overlap a direct-HIID row; these route totals can overlap. The coalesced-history counts are 39,948 with any X somewhere in the segment and 175,457 with direct-ID evidence somewhere in it. Of unique links, 174,639 are supported by one individual link row spanning both endpoints, while 1,220 need compatible adjacent or overlapping rows to span them.

The selected-format audit found no zero or out-of-range `SICH` values in the source rows; the missing/zero handling therefore does not change this sample. The final sample also matches A1 one-for-one on all common non-route fields: the same 139,737 `GVKEY`/`FYEAR` keys and the same EPS, price, return, date, currency, and link inputs. All final prices use the USD CRSP `PRC` path and all retained Compustat reporting currencies are USD. Before trimming, 25,661 target firm-years report CAD: 12,874 have a matching Compustat price/currency path and 12,787 do not; the matching CAD rows do not enter the final sample. USD source rows are 292,723, with 173,155 having a usable prior-date CRSP quote. No daily/monthly security-price currency conflict appears in the source audit.

Every final Return has a DSF observation and valid daily `RET` for every expected DSI session (`nday=expdays`); the session count ranges from 20 to 485, with a median of 253. Among uniquely linked pretrim windows, 11,341 have a missing or invalid daily `RET`, 15 lack the first expected session, 65 lack the last expected session, and 256 have no DSF observations. End gaps may include early delistings, but the source does not specify a delisting-return extension, so no `DLRET` is added and such windows remain unavailable. The final actual `DATADATE` spans range from 30 to 701 days (median 365); 390 final rows are shorter than 300 days and 584 exceed 400 days. Before trimming these counts are 408 and 624. The 300–400 day boundaries are descriptive labels, not sample screens. For prior-date USD prices the quote-age median is zero, the 99th percentile is two days, and the maximum is six days under the seven-day rule.

The constructed complete-case sample before trimming is 145,055, which is 8,168 above the paper's initial 136,887 annual firm-years. After within-year 1st/99th-percentile trimming, N is 139,737, 7,817 above the published 131,920. The point estimate is lower than the reported 0.302 by 0.0182694. These are observed differences; the current merged Compustat/CRSP vintage, timing choice, source rows, and empirical screens differ from the paper's historical data, and the available evidence does not isolate one cause.

Stata's built-in pooled OLS with conventional standard errors estimates the interaction at 0.2837306 (SE 0.0034418, t=82.44, two-sided p<0.001, R-squared 0.1211, N=139,737). The paper does not report an inference method for this annual fit. The standalone template outputs are in `tables/annual_pooled_eq1.{csv,md,rtf,tex}` and `tables/annual_descriptives.{csv,md,rtf,tex}`. Detailed independent checks and all coefficient rows are in `review.md`.

## Estimation and outputs

The source calls the pooled Equation (1) estimate a regression but does not identify the estimator, fixed effects, weights, or standard-error method. I will fit the pooled linear equation with Stata's built-in `regress` and its conventional homoskedastic standard errors. This is a best-effort estimator/inference choice; no fixed effects, weights, or clustering are introduced. Stata's two-sided p-values will be reported under the template's formatting.

The annual interaction coefficient is the research-interest estimate. The baseline table is exported in the template's four formats, with descriptive statistics for the actual retained analysis sample. The published 131,920 N and 0.302 coefficient are compared in `review.md`; no filter, date, or adjustment was changed to pursue either target.

## Known limits

- The pooled annual return-window choice and stock-return detail are not fit-specific in the paper; fiscal-year daily CRSP `RET` is the documented best-effort selection here.
- OLS and conventional standard errors are not explicitly identified for the paper's pooled annual fit.
- The modern Compustat/CRSP tables may differ from the Compustat 2001 files and return vintage; the fit is not tuned to source N or coefficients.
- The provider's basic/primary EPS convention changed in 1997. EPSPX/AJEX matches the item definition available in the current local product, but cannot restore the historical 2001 data snapshot.
- Currency mismatches or missing period-specific security currency codes are left unavailable. The CRSP return remains a dimensionless USD-market holding-period return. The ratio uses a USD CRSP quote for USD reporting EPS, or a Compustat annual close when matched security-price currency agrees with non-USD reporting EPS; no dollar-rate conversion is inferred from a translation-rate field.
- The target scope is FY 1951–2001. Historical `FUNDA.HIID` matching and dated, provider-documented X-placeholder CCM links are used to assess actual coverage. No backdated CCM link, current CUSIP, or name match is used.


In [ ]:
options errorabend nodate nonumber;
libname comp '/storage/sas/base/comp/d_na' access=readonly;
libname compsec '/storage/sas/base/comp/d_na/security' access=readonly;
libname crsp '/storage/sas/base/crsp/a_ccm' access=readonly;
libname crspstk '/storage/sas/base/crsp/a_stock' access=readonly;
libname out 'data';

data work.funda_raw;
    length pref_fmt $4;
    set comp.funda(keep=gvkey datadate fyear indfmt datafmt consol popsrc
                   iid hiid curcd epspx ajex prcc_f sich naicsh
                   where=(fyear >= 1950 and fyear <= 2001
                          and popsrc='D' and datafmt='STD' and consol='C'
                          and indfmt in ('INDL','FS')));
    sich_valid=(not missing(sich) and 100<=sich and sich<=9999);
    sich_zero=(sich=0);
    naicsh_zero=(naicsh=0);
    naics_sector=.;
    if not missing(naicsh) and naicsh>=10 then do;
        naics_digits=floor(log10(naicsh))+1;
        naics_sector=floor(naicsh/(10**(naics_digits-2)));
    end;
    if sich_valid and 6000 <= sich and sich <= 6999 then pref_fmt='FS';
    else if sich_valid then pref_fmt='INDL';
    else if naics_sector=52 then pref_fmt='FS';
    else if not missing(naics_sector) then pref_fmt='INDL';
    else pref_fmt='';
    if indfmt=pref_fmt and not missing(pref_fmt) then fmtmatch=1;
    else fmtmatch=0;
run;

proc sql;
    create table work.rawcounts as
    select count(*) as filtered_source_rows,
           max(lengthn(strip(gvkey))) as max_gvkey_chars
    from work.funda_raw;
quit;

proc sort data=work.funda_raw out=work.funda_uniq nodupkey;
    by gvkey fyear datadate iid hiid curcd indfmt epspx ajex prcc_f sich naicsh;
run;
proc sort data=work.funda_raw out=work.valrows nodupkey;
    by gvkey fyear datadate iid hiid curcd epspx ajex prcc_f sich naicsh;
run;
data work.prefrows;
    set work.funda_uniq;
    if fmtmatch=1;
run;
proc sort data=work.prefrows nodupkey;
    by gvkey fyear datadate iid hiid curcd epspx ajex prcc_f sich naicsh;
run;

proc sql;
    create table work.prefstats as
    select gvkey, fyear, count(*) as preferred_rows
    from work.prefrows
    group by gvkey, fyear;

    create table work.fystats as
    select v.gvkey, v.fyear, min(v.datadate) as firstdate,
           max(v.datadate) as lastdate, count(*) as candidates,
           coalesce(p.preferred_rows,0) as preferred_rows
    from work.valrows as v
    left join work.prefstats as p
      on v.gvkey=p.gvkey and v.fyear=p.fyear
    group by v.gvkey, v.fyear, p.preferred_rows;

    create table work.annual0 as
    select v.*
    from work.valrows as v
    inner join work.fystats as s
      on v.gvkey=s.gvkey and v.fyear=s.fyear
    where s.candidates=1
    union corr
    select p.*
    from work.prefrows as p
    inner join work.fystats as s
      on p.gvkey=s.gvkey and p.fyear=s.fyear
    where s.candidates>1 and s.firstdate=s.lastdate
      and s.preferred_rows=1;

    create table work.fycounts as
    select count(*) as firmyears_selected
    from work.annual0;

    create table work.ambig as
    select count(*) as firmyears_ambiguous
    from work.fystats
    where candidates>1 and
          (firstdate ne lastdate or preferred_rows ne 1);

    create table work.ambigyr as
    select fyear, count(*) as ambiguous_firmyears
    from work.fystats
    where candidates>1 and
          (firstdate ne lastdate or preferred_rows ne 1)
    group by fyear
    order by fyear;

    create table work.indcov as
    select fyear, count(*) as source_rows,
           sum(sich_valid) as valid_sich_rows,
           sum(sich_zero) as zero_sich_rows,
           sum(not missing(sich) and sich_valid=0 and sich_zero=0)
               as other_invalid_sich_rows,
           sum(not missing(naicsh) and naicsh>0) as positive_naicsh_rows,
           sum(naicsh_zero) as zero_naicsh_rows,
           sum(naics_sector=52) as naics_finance_sector_rows
    from work.valrows
    group by fyear
    order by fyear;
quit;

data work.annual;
    set work.annual0;
    source_fmt=indfmt;
    keep gvkey fyear datadate iid hiid curcd indfmt source_fmt sich naicsh
         epspx ajex prcc_f;
run;

proc sql;
    create table work.period0 as
    select a.gvkey, a.fyear, a.datadate, a.iid, a.hiid, a.curcd,
           a.indfmt, a.sich, a.naicsh, a.epspx, a.ajex, a.prcc_f,
           b.datadate as prevdate, b.iid as previid, b.hiid as prevhiid,
           b.curcd as prevcur, b.ajex as prevajex,
           b.prcc_f as prevprcc
    from work.annual as a
    left join work.annual as b
      on a.gvkey=b.gvkey and a.fyear=b.fyear+1
    where a.fyear between 1951 and 2001;
quit;

data work.period0;
    set work.period0;
    lagok=(not missing(prevdate));
    dateok=(lagok and prevdate<datadate);
    iidok=(lagok and strip(iid)=strip(previid));
    hiidok=(lagok and not missing(hiid) and not missing(prevhiid)
            and strip(hiid)=strip(prevhiid));
    epsok=(not missing(epspx) and not missing(ajex) and ajex>0
           and not missing(prevajex) and prevajex>0);
    ret_days=datadate-prevdate;
    eps_prior_basis=.;
    if epsok then eps_prior_basis=epspx*prevajex/ajex;
run;

data work.issues;
    length iid $4;
    set work.period0(keep=gvkey hiid lagok dateok hiidok);
    if lagok and dateok and hiidok;
    iid=hiid;
    keep gvkey iid;
run;
proc sort data=work.issues nodupkey;
    by gvkey iid;
run;

data work.links;
    length iid $4 sdate edate 8 is_x 8;
    set crsp.ccmxpf_linktable(keep=gvkey liid lpermno linkdt linkenddt
                              linkprim linktype USEDFLAG);
    if USEDFLAG ne 1 or linkprim not in ('P','C')
       or linktype not in ('LC','LU')
       or missing(lpermno) then delete;
    iid=liid;
    is_x=0;
    if not missing(iid)
       and substr(strip(upcase(iid)),lengthn(strip(upcase(iid))),1)='X'
       then is_x=1;
    if linkdt=.B then sdate=-1000000;
    else if not missing(linkdt) then sdate=linkdt;
    else delete;
    if linkenddt=.E or linkenddt=. then edate=1000000;
    else if not missing(linkenddt) then edate=linkenddt;
    else delete;
    if edate<sdate then delete;
    keep gvkey iid lpermno linkdt linkenddt linkprim linktype USEDFLAG
         sdate edate is_x;
run;
proc sql;
    create table work.linkintervals as
    select distinct p.gvkey, p.fyear, p.hiid, p.prevdate, p.datadate,
           l.lpermno as permno, l.sdate, l.edate, l.is_x,
           (l.iid=p.hiid) as direct_id
    from work.period0 as p
    inner join work.links as l
      on p.gvkey=l.gvkey and (l.iid=p.hiid or l.is_x=1)
    where p.lagok=1 and p.dateok=1 and p.hiidok=1;

    create table work.rowcand as
    select distinct p.gvkey, p.fyear, l.lpermno as permno
    from work.period0 as p
    inner join work.links as l
      on p.gvkey=l.gvkey and (l.iid=p.hiid or l.is_x=1)
    where p.lagok=1 and p.dateok=1 and p.hiidok=1
      and l.sdate<=p.prevdate and l.edate>=p.datadate;
quit;

proc sort data=work.linkintervals;
    by gvkey fyear hiid permno sdate edate;
run;
data work.linkseg;
    set work.linkintervals;
    by gvkey fyear hiid permno;
    retain segstart segend dummy_x_history direct_id_history;
    if first.permno then do;
        segstart=sdate;
        segend=edate;
        dummy_x_history=is_x;
        direct_id_history=direct_id;
    end;
    else if sdate<=segend+1 then do;
        if edate>segend then segend=edate;
        dummy_x_history=max(dummy_x_history,is_x);
        direct_id_history=max(direct_id_history,direct_id);
    end;
    else do;
        output;
        segstart=sdate;
        segend=edate;
        dummy_x_history=is_x;
        direct_id_history=direct_id;
    end;
    if last.permno then output;
    keep gvkey fyear hiid permno segstart segend
         dummy_x_history direct_id_history;
run;

proc sql;
    create table work.linkcand0 as
    select p.gvkey, p.fyear, l.permno,
           l.dummy_x_history, l.direct_id_history,
           max(k.is_x) as dummy_x_used,
           max(case when k.iid=p.hiid then 1 else 0 end) as direct_id_used
    from work.period0 as p
    inner join work.linkseg as l
      on p.gvkey=l.gvkey and p.fyear=l.fyear and p.hiid=l.hiid
    inner join work.links as k
      on p.gvkey=k.gvkey and l.permno=k.lpermno
     and (k.iid=p.hiid or k.is_x=1)
     and k.sdate<=p.datadate and k.edate>=p.prevdate
    where p.lagok=1 and p.dateok=1 and p.hiidok=1
      and l.segstart<=p.prevdate and l.segend>=p.datadate
    group by p.gvkey, p.fyear, l.permno,
             l.dummy_x_history, l.direct_id_history;

    create table work.rowlinkstats as
    select gvkey, fyear, count(*) as rowlinkcnt
    from work.rowcand
    group by gvkey, fyear;

    create table work.linkcand as
    select gvkey, fyear, permno, max(dummy_x_used) as dummy_x_used,
           max(direct_id_used) as direct_id_used,
           max(dummy_x_history) as dummy_x_history,
           max(direct_id_history) as direct_id_history
    from work.linkcand0
    group by gvkey, fyear, permno;

    create table work.linkstats as
    select gvkey, fyear, count(*) as linkcnt, max(permno) as permno,
           max(dummy_x_used) as dummy_x_used,
           max(direct_id_used) as direct_id_used,
           max(dummy_x_history) as dummy_x_history,
           max(direct_id_history) as direct_id_history
    from work.linkcand
    group by gvkey, fyear;
quit;

proc sql;
    create table work.periodlink as
    select p.*, coalesce(l.linkcnt,0) as linkcnt,
           coalesce(r.rowlinkcnt,0) as rowlinkcnt,
           coalesce(l.dummy_x_used,0) as dummy_x_used,
           coalesce(l.direct_id_used,0) as direct_id_used,
           coalesce(l.dummy_x_history,0) as dummy_x_history,
           coalesce(l.direct_id_history,0) as direct_id_history,
           l.permno as permno
    from work.period0 as p
    left join work.linkstats as l
      on p.gvkey=l.gvkey and p.fyear=l.fyear
    left join work.rowlinkstats as r
      on p.gvkey=r.gvkey and p.fyear=r.fyear;
quit;

data work.win;
    set work.periodlink;
    if lagok and dateok and hiidok and linkcnt=1;
    keep gvkey fyear iid hiid curcd prevdate datadate ret_days permno;
run;
proc sort data=work.win;
    by gvkey fyear;
run;

data work.marketcal;
    length date 8;
    set crspstk.dsi(keep=date
                    where=(date >= '01JAN1948'd
                           and date <= '31DEC2003'd));
run;
proc sort data=work.marketcal nodupkey;
    by date;
run;
data work.marketcal;
    set work.marketcal;
    mktday+1;
run;

data work.events;
    length date 8;
    set work.win;
    which=1;
    date=prevdate;
    output;
    which=2;
    date=datadate;
    output;
    keep gvkey fyear which date;
run;
proc sort data=work.events;
    by date;
run;
data work.anchor_rank;
    merge work.marketcal(in=im) work.events(in=ie);
    by date;
    retain lastday lastdate;
    if im then do;
        lastday=mktday;
        lastdate=date;
    end;
    if ie then do;
        eventrank=lastday;
        eventmkt=lastdate;
        output;
    end;
    keep gvkey fyear which eventrank eventmkt;
run;
proc sort data=work.anchor_rank;
    by gvkey fyear which;
run;
data work.marketcnt;
    set work.anchor_rank;
    by gvkey fyear;
    retain rprev rcurr dprev dcurr;
    if first.fyear then call missing(rprev,rcurr,dprev,dcurr);
    if which=1 then do;
        rprev=eventrank;
        dprev=eventmkt;
    end;
    else do;
        rcurr=eventrank;
        dcurr=eventmkt;
    end;
    if last.fyear then do;
        expdays=rcurr-rprev;
        output;
    end;
    keep gvkey fyear rprev rcurr dprev dcurr expdays;
run;
proc sql;
    create table work.marketwin as
    select m.*, a.date as exp_first, b.date as exp_last
    from work.marketcnt as m
    left join work.marketcal as a
      on a.mktday=m.rprev+1
    left join work.marketcal as b
      on b.mktday=m.rcurr;
quit;

data work.permnos;
    set work.win(keep=permno);
run;
proc sort data=work.permnos nodupkey;
    by permno;
run;

data work.dsf0;
    if _n_=1 then do;
        declare hash hp(dataset:'work.permnos');
        hp.defineKey('permno');
        hp.defineDone();
    end;
    set crspstk.dsf(keep=permno date prc ret
                    where=(date >= '01JAN1948'd
                           and date <= '31DEC2003'd));
    if hp.check()=0;
run;
proc sort data=work.dsf0;
    by permno date;
run;
data work.dsfuniq work.dsfdup;
    set work.dsf0;
    by permno date;
    if first.date and last.date then output work.dsfuniq;
    else if first.date then output work.dsfdup;
run;

data work.dayret;
    length gvkey $7 fyear 8 prevdate datadate 8 rc 8;
    if _n_=1 then do;
        declare hash hw(dataset:'work.win',multidata:'y');
        hw.defineKey('permno');
        hw.defineData('gvkey','fyear','prevdate','datadate');
        hw.defineDone();
    end;
    set work.dsfuniq(keep=permno date ret);
    rc=hw.find();
    do while (rc=0);
        if date>prevdate and date<=datadate then output;
        rc=hw.find_next();
    end;
    keep gvkey fyear permno date ret;
run;
proc sort data=work.dayret;
    by gvkey fyear date;
run;

data work.retagg;
    set work.dayret;
    by gvkey fyear;
    retain nday ret_bad logret zero_ret first_obs last_obs;
    if first.fyear then do;
        nday=0;
        ret_bad=0;
        logret=0;
        zero_ret=0;
        first_obs=date;
    end;
    last_obs=date;
    nday+1;
    if missing(ret) or ret < -1 then ret_bad=1;
    else if ret=-1 then zero_ret=1;
    else logret=logret+log(1+ret);
    if last.fyear then do;
        ret_annual=.;
        if nday>0 and ret_bad=0 then do;
            if zero_ret=1 then ret_annual=-1;
            else ret_annual=exp(logret)-1;
        end;
        output;
    end;
    keep gvkey fyear nday ret_bad ret_annual first_obs last_obs;
run;
proc sort data=work.retagg;
    by gvkey fyear;
run;
proc sort data=work.marketwin;
    by gvkey fyear;
run;
data work.retagg2;
    merge work.retagg(in=a) work.marketwin(in=b);
    by gvkey fyear;
    if a or b;
    ret=.;
    retok=0;
    if a and b and not missing(expdays) and expdays>0
       and nday=expdays and first_obs=exp_first
       and last_obs=exp_last and ret_bad=0 then do;
        ret=ret_annual;
        retok=1;
    end;
    keep gvkey fyear ret nday ret_bad retok expdays exp_first exp_last
         first_obs last_obs;
run;

data work.qwin;
    set work.win;
    if strip(upcase(curcd))='USD';
    keep gvkey fyear permno prevdate;
run;
data work.quotehit;
    length gvkey $7 fyear 8 prevdate 8 qdate 8 qgap 8 qprice 8 rc 8;
    if _n_=1 then do;
        declare hash hq(dataset:'work.qwin',multidata:'y');
        hq.defineKey('permno');
        hq.defineData('gvkey','fyear','prevdate');
        hq.defineDone();
    end;
    set work.dsfuniq(keep=permno date prc);
    rc=hq.find();
    do while (rc=0);
        if date<=prevdate and date>=prevdate-7 and not missing(prc) then do;
            qdate=date;
            qgap=prevdate-date;
            qprice=abs(prc);
            output;
        end;
        rc=hq.find_next();
    end;
    keep gvkey fyear qdate qgap qprice;
run;
proc sort data=work.quotehit;
    by gvkey fyear descending qdate;
run;
data work.qbest;
    set work.quotehit;
    by gvkey fyear;
    if first.fyear;
    rename qdate=crsp_date qgap=crsp_gap qprice=crsp_price;
run;

proc sql;
    create table work.dupflag as
    select w.gvkey, w.fyear,
           max(case when d.date>w.prevdate and d.date<=w.datadate
                    then 1 else 0 end) as retdup,
           max(case when d.date<=w.prevdate and d.date>=w.prevdate-7
                    then 1 else 0 end) as pricedup
    from work.win as w
    inner join work.dsfdup as d
      on w.permno=d.permno
    group by w.gvkey, w.fyear;
quit;

data work.dcur0;
    if _n_=1 then do;
        declare hash hi(dataset:'work.issues');
        hi.defineKey('gvkey','iid');
        hi.defineDone();
    end;
    set compsec.sec_dprc(keep=gvkey iid datadate curcdd
                         where=(datadate >= '01JAN1948'd
                                and datadate <= '31DEC2003'd));
    if hi.check()=0 and not missing(curcdd);
run;
proc sort data=work.dcur0 out=work.dcur1 nodupkey;
    by gvkey iid datadate curcdd;
run;
proc sql;
    create table work.dcurcnt as
    select gvkey, iid, datadate, count(*) as ncur
    from work.dcur1
    group by gvkey, iid, datadate;

    create table work.dcur as
    select a.gvkey, a.iid, a.datadate, a.curcdd
    from work.dcur1 as a
    inner join work.dcurcnt as c
      on a.gvkey=c.gvkey and a.iid=c.iid and a.datadate=c.datadate
    where c.ncur=1;
quit;

data work.dreq;
    length iid $4;
    set work.period0;
    if lagok and dateok and hiidok;
    iid=hiid;
    do doff=0 to 7;
        datadate=prevdate-doff;
        output;
    end;
    keep gvkey iid fyear prevdate doff datadate;
run;
proc sort data=work.dreq;
    by gvkey iid datadate;
run;
proc sort data=work.dcur;
    by gvkey iid datadate;
run;
data work.dhit;
    merge work.dreq(in=a) work.dcur(in=b);
    by gvkey iid datadate;
    if a and b;
run;
proc sort data=work.dhit;
    by gvkey fyear doff;
run;
data work.dbest;
    set work.dhit;
    by gvkey fyear;
    if first.fyear;
    rename datadate=secday_date curcdd=daycur doff=secday_gap;
run;

data work.mcur0;
    if _n_=1 then do;
        declare hash hi(dataset:'work.issues');
        hi.defineKey('gvkey','iid');
        hi.defineDone();
    end;
    set compsec.sec_mthprc(keep=gvkey iid datadate curcdm
                           where=(datadate >= '01JAN1948'd
                                  and datadate <= '31DEC2003'd));
    if hi.check()=0 and not missing(curcdm);
    monid=intck('month','01JAN1948'd,datadate);
run;
proc sort data=work.mcur0 out=work.mcur1 nodupkey;
    by gvkey iid monid datadate curcdm;
run;
proc sql;
    create table work.mcurmax as
    select gvkey, iid, monid, max(datadate) as datadate
    from work.mcur1
    group by gvkey, iid, monid;

    create table work.mcurcnt as
    select a.gvkey, a.iid, a.monid, a.datadate, count(*) as ncur
    from work.mcur1 as a
    inner join work.mcurmax as m
      on a.gvkey=m.gvkey and a.iid=m.iid
     and a.monid=m.monid and a.datadate=m.datadate
    group by a.gvkey, a.iid, a.monid, a.datadate;

    create table work.mcur as
    select a.gvkey, a.iid, a.monid, a.datadate, a.curcdm
    from work.mcur1 as a
    inner join work.mcurcnt as c
      on a.gvkey=c.gvkey and a.iid=c.iid and a.monid=c.monid
     and a.datadate=c.datadate
    where c.ncur=1;
quit;

data work.mreq;
    length iid $4;
    set work.period0;
    if lagok and dateok and hiidok;
    iid=hiid;
    targetmon=intck('month','01JAN1948'd,prevdate);
    do moff=0 to 1;
        monid=targetmon-moff;
        output;
    end;
    keep gvkey iid fyear prevdate moff monid;
run;
proc sort data=work.mreq;
    by gvkey iid monid;
run;
proc sort data=work.mcur;
    by gvkey iid monid;
run;
data work.mhit;
    merge work.mreq(in=a) work.mcur(in=b);
    by gvkey iid monid;
    if a and b and datadate<=prevdate
       and prevdate-datadate<=45;
    mth_gap=prevdate-datadate;
run;
proc sort data=work.mhit;
    by gvkey fyear moff;
run;
data work.mbest;
    set work.mhit;
    by gvkey fyear;
    if first.fyear;
    rename datadate=secmth_date curcdm=mthcur;
run;

proc sql;
    create table work.stage0 as
    select p.*, coalesce(l.linkcnt,0) as linkcnt,
           coalesce(rl.rowlinkcnt,0) as rowlinkcnt,
           coalesce(l.dummy_x_used,0) as dummy_x_used,
           coalesce(l.direct_id_used,0) as direct_id_used,
           coalesce(l.dummy_x_history,0) as dummy_x_history,
           coalesce(l.direct_id_history,0) as direct_id_history,
           l.permno, r.nday, r.ret_bad, r.ret, r.retok,
           r.expdays, r.exp_first, r.exp_last, r.first_obs, r.last_obs,
           q.crsp_date, q.crsp_gap, q.crsp_price,
           x.retdup, x.pricedup,
           d.secday_date, d.secday_gap, d.daycur,
           m.secmth_date, m.mth_gap, m.mthcur
    from work.period0 as p
    left join work.linkstats as l
      on p.gvkey=l.gvkey and p.fyear=l.fyear
    left join work.rowlinkstats as rl
      on p.gvkey=rl.gvkey and p.fyear=rl.fyear
    left join work.retagg2 as r
      on p.gvkey=r.gvkey and p.fyear=r.fyear
    left join work.qbest as q
      on p.gvkey=q.gvkey and p.fyear=q.fyear
    left join work.dupflag as x
      on p.gvkey=x.gvkey and p.fyear=x.fyear
    left join work.dbest as d
      on p.gvkey=d.gvkey and p.fyear=d.fyear
    left join work.mbest as m
      on p.gvkey=m.gvkey and p.fyear=m.fyear;
quit;

data work.stage1;
    length price_cur epscur $3 price_source $4;
    set work.stage0;
    if missing(linkcnt) then linkcnt=0;
    if missing(dummy_x_used) then dummy_x_used=0;
    if missing(direct_id_used) then direct_id_used=0;
    if missing(dummy_x_history) then dummy_x_history=0;
    if missing(direct_id_history) then direct_id_history=0;
    if missing(retdup) then retdup=0;
    if missing(pricedup) then pricedup=0;
    if retdup=1 then ret=.;
    if missing(ret_bad) then ret_bad=0;
    retok=(retok=1 and not missing(ret) and ret_bad=0 and retdup=0
           and nday=expdays and first_obs=exp_first
           and last_obs=exp_last);
    price_cur='';
    price=.;
    price_date=.;
    price_gap=.;
    price_source='';
    cur_conflict=0;
    curmatch=0;
    if not missing(curcd) then do;
        epscur=strip(upcase(curcd));
        daycur=strip(upcase(daycur));
        mthcur=strip(upcase(mthcur));
        if not missing(daycur) and not missing(mthcur)
           and daycur ne mthcur then cur_conflict=1;
        if not missing(daycur) and not missing(mthcur)
           and daycur=mthcur then price_cur=daycur;
        else if not missing(daycur) and missing(mthcur) then price_cur=daycur;
        else if missing(daycur) and not missing(mthcur) then price_cur=mthcur;
        if epscur='USD' then do;
            price_source='CRSP';
            price_cur='USD';
            if pricedup=0 and not missing(crsp_price) and crsp_price>0 then do;
                price=crsp_price;
                price_date=crsp_date;
                price_gap=crsp_gap;
                curmatch=1;
            end;
        end;
        else if cur_conflict=0 and not missing(price_cur)
             and price_cur=epscur and not missing(prevprcc)
             and prevprcc>0 then do;
            price_source='COMP';
            price=prevprcc;
            price_date=prevdate;
            price_gap=0;
            curmatch=1;
        end;
    end;
    priceok=(curmatch=1 and not missing(price) and price>0
             and not missing(eps_prior_basis));
    eps_price=.;
    if priceok then eps_price=eps_prior_basis/price;
    dum=.;
    dum_return=.;
    if not missing(ret) then do;
        dum=(ret<0);
        dum_return=dum*ret;
    end;
    complete=(not missing(eps_price) and not missing(ret)
              and not missing(dum) and not missing(dum_return));
run;

proc sort data=work.stage1;
    by fyear;
run;
proc univariate data=work.stage1 noprint;
    where complete=1;
    by fyear;
    var eps_price ret;
    output out=work.cutoffs pctlpts=1 99 pctlpre=eps_ ret_;
run;
data work.final0;
    merge work.stage1(in=a) work.cutoffs;
    by fyear;
    if a and complete=1;
    if eps_1<=eps_price and eps_price<=eps_99
       and ret_1<=ret and ret<=ret_99;
run;
data work.final;
    set work.final0;
    keep gvkey fyear iid previid hiid prevhiid hiidok curcd prevcur indfmt sich naicsh
         datadate prevdate ret_days epspx ajex prevajex eps_prior_basis
         prevprcc price price_source price_cur price_date price_gap
         secday_date secday_gap secmth_date mth_gap crsp_date
         crsp_gap linkcnt rowlinkcnt dummy_x_used direct_id_used
         dummy_x_history direct_id_history permno
         nday expdays exp_first exp_last
         first_obs last_obs ret_bad ret retok retdup pricedup
         cur_conflict curmatch eps_price dum dum_return;
run;

proc sql;
    create table work.attrsum as
    select count(*) as source_firmyears,
           sum(lagok) as adjacent_lags,
           sum(dateok) as ordered_actual_dates,
           sum(iidok) as stable_header_iid,
           sum(hiidok) as stable_historical_primary_hiid,
           sum(epsok) as eps_and_split_inputs,
           sum(linkcnt=1) as unique_crsp_links,
           sum(rowlinkcnt=1) as unique_single_row_links,
           sum(linkcnt=1 and rowlinkcnt=0) as segment_only_unique_links,
           sum(linkcnt=1 and dummy_x_used=1) as unique_links_using_dummy_x,
           sum(linkcnt=1 and direct_id_used=1) as unique_links_using_hiid,
           sum(linkcnt=1 and dummy_x_history=1) as unique_links_x_history,
           sum(linkcnt=1 and direct_id_history=1) as unique_links_hiid_history,
           sum(retok) as complete_daily_returns,
           sum(curmatch=1) as matched_currency_and_price,
           sum(complete) as complete_regression_rows
    from work.stage1;

    create table work.attryear as
    select fyear, count(*) as source_rows,
           sum(lagok) as lag_rows,
           sum(iidok) as same_header_iid,
           sum(hiidok) as same_historical_primary_hiid,
           sum(linkcnt=1) as unique_links,
           sum(rowlinkcnt=1) as unique_single_row_links,
           sum(linkcnt=1 and rowlinkcnt=0) as segment_only_unique_links,
           sum(linkcnt=1 and dummy_x_used=1) as unique_links_using_dummy_x,
           sum(linkcnt=1 and direct_id_used=1) as unique_links_using_hiid,
           sum(linkcnt=1 and dummy_x_history=1) as unique_links_x_history,
           sum(linkcnt=1 and direct_id_history=1) as unique_links_hiid_history,
           sum(retok) as valid_return,
           sum(priceok) as valid_price,
           sum(complete) as complete_rows
    from work.stage1
    group by fyear
    order by fyear;

    create table work.tailcount as
    select count(*) as final_firmyears
    from work.final;
quit;

title 'Annual source selection and construction attrition';
proc print data=work.rawcounts noobs;
run;
proc print data=work.fycounts noobs;
run;
proc print data=work.ambig noobs;
run;
proc print data=work.ambigyr noobs;
run;
proc print data=work.indcov noobs;
run;
proc print data=work.attrsum noobs;
run;
proc print data=work.tailcount noobs;
run;
proc print data=work.attryear noobs;
run;
title 'Retained source paths and currencies';
proc freq data=work.final;
    tables fyear*price_source curcd*price_cur / missing;
    tables dummy_x_used*direct_id_used
           dummy_x_history*direct_id_history / missing;
run;
title 'Currency and prior-price availability before annual trimming';
proc freq data=work.stage1;
    tables curcd*curmatch curcd*price_source cur_conflict / missing;
run;
title 'Actual prior-to-current return-window lengths before annual trimming';
data work.span_pre;
    length span_class $16;
    set work.stage1;
    if complete=1;
    if ret_days<300 then span_class='short (<300d)';
    else if ret_days>400 then span_class='long (>400d)';
    else span_class='300-400d';
run;
proc freq data=work.span_pre;
    tables span_class / missing;
run;
title 'Actual prior-to-current return-window lengths after annual trimming';
data work.span_final;
    length span_class $16;
    set work.final0;
    if ret_days<300 then span_class='short (<300d)';
    else if ret_days>400 then span_class='long (>400d)';
    else span_class='300-400d';
run;
proc freq data=work.span_final;
    tables span_class / missing;
run;
title 'Return-window coverage and missing or invalid daily returns';
data work.return_audit;
    length return_status $38;
    set work.stage1;
    if linkcnt ne 1 then return_status='No unique date-valid PERMNO';
    else if missing(expdays) or expdays<=0 then return_status='No valid DSI window';
    else if missing(nday) then return_status='No DSF daily observations';
    else if retdup=1 then return_status='Duplicate DSF PERMNO-date';
    else if last_obs ne exp_last then return_status='No DSF at last expected day';
    else if first_obs ne exp_first then return_status='No DSF at first expected day';
    else if nday<expdays then return_status='Interior DSI-session gap';
    else if nday>expdays then return_status='Extra DSF session in window';
    else if ret_bad=1 then return_status='Missing or invalid daily RET';
    else if retok=1 then return_status='Complete daily RET window';
    else return_status='Other incomplete return window';
run;
proc freq data=work.return_audit;
    tables return_status / missing;
run;
title 'Final analysis variable and return coverage distributions';
proc means data=work.final n nmiss mean std min p1 p50 p99 max maxdec=6;
    var epspx ajex prevajex eps_prior_basis prevprcc price eps_price
        ret ret_days nday expdays price_gap crsp_gap secday_gap mth_gap;
run;
title 'SAS output sample structure';
proc contents data=work.final;
run;
title;

data out.reg;
    set work.final;
    label eps_price='EPSPX rebased to prior share basis / prior price'
          ret='Compounded raw CRSP daily total return'
          dum='Indicator for negative Return'
          dum_return='DUM times Return'
          ret_days='Days between actual adjacent DATADATEs';
    format datadate prevdate price_date crsp_date
           secday_date secmth_date exp_first exp_last
           first_obs last_obs yymmdd10.;
run;

%put FULL_BUILD_DONE;
